# Linear Regression — Optimization

**Goal.** Replace the closed-form OLS solution from `02_mathematics.ipynb` with an **iterative algorithm** — gradient descent and its stochastic variants — that scales to large $p$, works for any differentiable loss, and admits a clean convergence theory.

**Role of this notebook.** *Algorithms* — pseudocode plus minimal demo code. The loss math (gradient, Hessian, convexity) is in `02_mathematics.ipynb`; the visual intuition in `01_intuition.ipynb`; the full from-scratch implementation in `05_hands_on_programming.ipynb`. Code here exists only to *exhibit* algorithm behaviour, not to be a reference implementation.

**Prerequisites.** `02_mathematics.ipynb` — in particular the gradient formula (3.1) $\nabla L(\theta) = \frac{2}{n} X^\top (X\theta - y)$, the convexity theorem 3.4, and the SVD of $X$ (6.1).

**Stage map.** `01_intuition` → `02_mathematics` → **`03_optimization`** → `04_statistics` → `05_hands_on_programming`.

**Six questions.**

1. Why iterate at all, given the closed form (4.2) of `02_mathematics`?
2. What is the gradient-descent update rule, formally?
3. What does GD *do* on the OLS bowl $L(\theta)$?
4. When does GD converge, and how fast?
5. What if we cannot afford the full gradient on each step? (SGD, mini-batch SGD.)
6. What practical levers control convergence? (Learning rate, feature scaling, stopping.)

---

**Reading conventions.** Same as `02_mathematics.ipynb`: theorem statements in blockquotes; multi-line derivations and pseudocode in code blocks; equations numbered (e.g. (2.1)) only when later cells refer back to them. Code cells exist to demonstrate algorithm behaviour, not derive math.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. Why iterative optimisation?

The closed-form OLS solution from Theorem 4.2 of `02_mathematics.ipynb`,

> $$\theta^* = (X^\top X)^{-1} X^\top y,$$

is exact but has three practical weaknesses.

1. **Cost.** Forming $(X^\top X)^{-1}$ (or equivalently solving the normal equations) is $\Theta(p^3)$ in the number of features $p$. For $p \approx 10^5$ (text, images) it is impractical.
2. **Memory.** $X^\top X$ is a $p \times p$ matrix. At $p = 10^5$ that is $10^{10}$ floats $\approx 80$ GB in float64.
3. **Generality.** The closed form only works for linear models with squared loss. Logistic loss, Huber loss, neural networks — no closed form exists at all.

**Gradient descent (GD)** is the workaround: instead of solving a linear system, walk downhill on the loss surface. Slower per pass but scales to large $p$ and works for almost any differentiable loss.

## 2. Gradient descent: the update rule

The entire algorithm is one equation.

> **Update rule (batch GD).**   $\theta_{k+1} = \theta_k - \eta \cdot \nabla L(\theta_k)$.   (2.1)

- $\nabla L(\theta_k)$ points in the direction of **steepest ascent**; negating it gives steepest descent.
- $\eta > 0$ is the **learning rate** (step size). Small $\eta$ → slow but safe. Large $\eta$ → fast but may overshoot or diverge. The safe range is governed by Theorem 4.2 below.

### 2.1 Algorithm (batch gradient descent)

```
ALGORITHM 1: Batch Gradient Descent

Input:   loss L : ℝᵖ → ℝ, gradient oracle ∇L,
         learning rate η > 0, initial θ₀ ∈ ℝᵖ,
         tolerance tol > 0, max iterations K.
Output:  approximate minimiser θ_K ≈ argmin L.

1.  for k = 0, 1, 2, …, K − 1:
2.      g_k ← ∇L(θ_k)
3.      if ‖g_k‖ ≤ tol:  return θ_k          ▷ stationary point reached
4.      θ_{k+1} ← θ_k − η · g_k
5.  return θ_K
```

Line 3 is sound *because* $L$ is convex (Theorem 3.4 of `02_mathematics.ipynb`): any stationary point is a global minimiser.

### 2.2 One-dimensional sanity demo

To build intuition, minimise $L(w) = (w - 3)^2$. The gradient is $2(w - 3)$, so the unique minimum is $w^* = 3$. Algorithm 1 should march to it.

In [ ]:
def loss_toy(w):
    return (w - 3.0) ** 2

def grad_toy(w):
    return 2.0 * (w - 3.0)

w = 0.0
eta = 0.1
trajectory = [w]
for _ in range(30):
    w = w - eta * grad_toy(w)
    trajectory.append(w)

print(f"Final w = {w:.4f}  (target w* = 3.0)")
print(f"Final L = {loss_toy(w):.6f}")

ws = np.linspace(-1, 7, 200)
plt.figure(figsize=(6, 4))
plt.plot(ws, loss_toy(ws), label="L(w)")
plt.plot(trajectory, [loss_toy(w_) for w_ in trajectory], "ro-", markersize=4, label="GD path")
plt.xlabel("w"); plt.ylabel("L(w)"); plt.legend()
plt.title("Algorithm 1 on a 1-D quadratic")
plt.show()

## 3. Gradient descent on the OLS bowl

For linear regression, the gradient is known in closed form (Theorem 3.2 of `02_mathematics.ipynb`):

> $$\nabla L(\theta) = \frac{2}{n} X^\top (X\theta - y). \quad (3.1)$$

Plugging (3.1) into the GD update (2.1):

$$\theta_{k+1} = \theta_k - \eta \cdot \frac{2}{n} X^\top (X\theta_k - y). \quad (3.2)$$

Each step costs $\Theta(np)$ — one matrix–vector product plus a subtraction — much cheaper than the $\Theta(p^3)$ closed form when $p$ is large.

### 3.1 Demo: fit $y = 2x + 1$ + noise

Same data as `02_mathematics.ipynb`. Initialise $\theta_0 = (0, 0)$; run 200 iterations of Algorithm 1 with $\eta = 0.05$. Two plots: the loss curve $L(\theta_k)$ and the **trajectory of $\theta_k$ on the 2-D bowl** of `02_mathematics.ipynb` §3.3 — this is the algorithm-specific picture, the one that makes "GD = walk downhill on the loss surface" visual.

In [ ]:
n = 100
x = rng.uniform(-3, 3, size=n)
noise = rng.normal(0, 0.5, size=n)
y = 2.0 * x + 1.0 + noise

X = np.column_stack([np.ones(n), x])

def mse(theta):
    return float(np.mean((X @ theta - y) ** 2))

def grad_mse(theta):
    return (2.0 / n) * X.T @ (X @ theta - y)

theta = np.zeros(2)
eta = 0.05
trajectory = [theta.copy()]
losses = [mse(theta)]
for _ in range(200):
    theta = theta - eta * grad_mse(theta)
    trajectory.append(theta.copy())
    losses.append(mse(theta))
trajectory = np.array(trajectory)

print(f"Estimated intercept = {theta[0]:.4f}   (true = 1.0)")
print(f"Estimated slope     = {theta[1]:.4f}   (true = 2.0)")
print(f"Final MSE           = {losses[-1]:.6f}")

# Two-panel plot: loss curve + trajectory on the 2-D OLS bowl.
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))

axes[0].plot(losses, color="steelblue")
axes[0].set_yscale("log")
axes[0].set_xlabel("step k"); axes[0].set_ylabel("L(θ_k)  (log scale)")
axes[0].set_title("Loss curve")

t0_grid = np.linspace(-0.4, 2.0, 80)
t1_grid = np.linspace(-0.4, 3.0, 80)
T0, T1 = np.meshgrid(t0_grid, t1_grid)
Theta_grid = np.stack([T0.ravel(), T1.ravel()], axis=0)
L_flat = np.mean((X @ Theta_grid - y[:, None]) ** 2, axis=0)
L_grid = L_flat.reshape(T0.shape)

cs = axes[1].contour(T0, T1, L_grid, levels=15, cmap="viridis")
axes[1].clabel(cs, inline=True, fontsize=7, fmt="%.1f")
axes[1].plot(trajectory[:, 0], trajectory[:, 1], "r-o", markersize=2.5, lw=1, alpha=0.75, label="GD trajectory")
axes[1].plot(trajectory[0, 0], trajectory[0, 1], "ko", markersize=8, label="θ₀")
axes[1].plot(trajectory[-1, 0], trajectory[-1, 1], "r*", markersize=15, label="θ_K")
axes[1].set_xlabel("intercept θ₀"); axes[1].set_ylabel("slope θ₁")
axes[1].set_title("GD trajectory on the OLS bowl")
axes[1].legend(loc="upper left")

plt.tight_layout(); plt.show()

## 4. When does GD converge, and how fast?

Two ingredients control convergence: **smoothness** (an upper bound on curvature) and **strong convexity** (a lower bound).

### 4.1 Definitions

> **Definition ($L$-smoothness).** $L$ is **$L$-smooth** if its gradient is $L$-Lipschitz: for some $L > 0$ and all $\theta_1, \theta_2 \in \mathbb{R}^p$,
> 
> $$\|\nabla L(\theta_1) - \nabla L(\theta_2)\| \le L \cdot \|\theta_1 - \theta_2\|.$$
> 
> Equivalently (for $C^2$ losses), $\nabla^2 L(\theta) \preceq L \cdot I$ for all $\theta$.

> **Definition ($\mu$-strong convexity).** $L$ is **$\mu$-strongly convex** ($\mu > 0$) if $\nabla^2 L(\theta) \succeq \mu \cdot I$ for all $\theta$. The ratio $\kappa := L / \mu$ is the **condition number** of the loss.

For our OLS loss, $\nabla^2 L = \frac{2}{n} X^\top X$ is constant (Theorem 3.3 of `02_mathematics.ipynb`), so the smoothness and strong-convexity constants are exactly the largest and smallest eigenvalues of the Hessian:

$$\begin{aligned}
L_{\text{smooth}} &= \lambda_{\max}(\nabla^2 L) = \frac{2}{n} \cdot \sigma_1(X)^2, \\
\mu_{\text{strong}} &= \lambda_{\min}(\nabla^2 L) = \frac{2}{n} \cdot \sigma_p(X)^2 \quad (\text{positive iff } \operatorname{rank}(X) = p), \\
\kappa &= L_{\text{smooth}} / \mu_{\text{strong}} = \bigl(\sigma_1(X) / \sigma_p(X)\bigr)^2 = \operatorname{cond}(X)^2.
\end{aligned}$$

($\sigma_1$, $\sigma_p$ are the largest and smallest singular values of $X$ — see Theorem 6.1 of `02_mathematics.ipynb`.)

### 4.2 Theorem (GD convergence rate)

> **Theorem 4.2.** Let $L$ be convex and $L_{\text{smooth}}$-smooth, with at least one minimiser $\theta^*$. Run Algorithm 1 with constant learning rate $\eta = 1 / L_{\text{smooth}}$ from any $\theta_0$. Then for every $k \ge 1$,
> 
> $$L(\theta_k) - L(\theta^*) \le \frac{1}{2k\eta} \cdot \|\theta_0 - \theta^*\|^2,$$
> 
> i.e. **sub-linear rate $O(1/k)$**. If additionally $L$ is $\mu$-strongly convex, the rate improves to **linear**:
> 
> $$\|\theta_k - \theta^*\| \le \left(\frac{\kappa - 1}{\kappa + 1}\right)^k \cdot \|\theta_0 - \theta^*\|, \quad \text{where } \kappa = L_{\text{smooth}} / \mu_{\text{strong}}.$$

*(Standard result in convex optimisation; see e.g. Nesterov, "Introductory Lectures on Convex Optimization", Ch. 2.)*

### 4.3 Three consequences

1. **Stable step size.** If $\eta > 2 / L_{\text{smooth}}$, even a strictly convex quadratic can diverge (overshoot at every step). The safe range is $0 < \eta \le 2 / L_{\text{smooth}}$, with $\eta = 1 / L_{\text{smooth}}$ optimal for the rate above.
2. **Condition number governs speed.** The linear-rate constant $(\kappa - 1)/(\kappa + 1)$ is fast when $\kappa \approx 1$ (well-conditioned) and slow when $\kappa \gg 1$ (ill-conditioned). Halving the gap takes about $(\kappa / 2) \cdot \ln 2$ steps.
3. **Feature scaling reduces $\kappa$.** Standardising features so columns of $X$ have comparable norms reduces $\sigma_1(X) / \sigma_p(X)$, hence $\kappa$. See §7.

## 5. The learning rate in practice

Theorem 4.2 says $\eta \in (0, 2 / L_{\text{smooth}}]$ is the convergent regime. In practice $L_{\text{smooth}}$ is rarely known exactly, so we tune $\eta$ empirically. Three regimes show up reliably.

- **$\eta$ too small** — loss falls but slowly; may not converge within the step budget.
- **$\eta$ just right** — fast, monotonic decrease toward the optimum.
- **$\eta$ too large** — loss oscillates or **increases**; the iterate flies past the minimum and diverges.

We confirm all three by sweeping $\eta$ over four orders of magnitude on the same OLS problem, and check the theoretical stable bound $2 / L_{\text{smooth}}$.

In [ ]:
etas = [0.001, 0.05, 0.3, 1.0]
n_steps = 100

plt.figure(figsize=(7, 4))
for eta_ in etas:
    theta = np.zeros(2)
    curve = []
    for _ in range(n_steps):
        theta = theta - eta_ * grad_mse(theta)
        curve.append(mse(theta))
    plt.plot(curve, label=f"η = {eta_}")
plt.yscale("log")
plt.xlabel("step k"); plt.ylabel("MSE (log scale)")
plt.title("Effect of learning rate on convergence")
plt.legend()
plt.show()

# Compare with the theoretical safe bound from Theorem 4.2.
sigma1 = np.linalg.svd(X, compute_uv=False)[0]
L_smooth = (2.0 / n) * sigma1 ** 2
print(f"σ₁(X)             = {sigma1:.4f}")
print(f"L_smooth = (2/n)·σ₁²  = {L_smooth:.4f}")
print(f"Stable LR bound  η_max = 2 / L_smooth = {2.0 / L_smooth:.4f}")
print(f"  → η = 1.0 is above the bound, so the orange curve in the plot blows up.")

## 6. Stochastic variants: when full gradients are too expensive

Algorithm 1 evaluates $\nabla L$ on **all** $n$ examples per step. For large $n$ (e.g. ImageNet, $n \approx 10^6$) even one full pass is unaffordable. The fix: estimate $\nabla L$ from a random subset.

The MSE loss already has the **sum** structure that makes this possible. Define the per-example loss

> $$\ell_i(\theta) := (x_i^\top \theta - y_i)^2,$$

so that $L(\theta) = \frac{1}{n} \sum_i \ell_i(\theta)$ and $\nabla L(\theta) = \frac{1}{n} \sum_i \nabla \ell_i(\theta)$. Any sum can be unbiasedly estimated by averaging a random sample.

### 6.1 Stochastic gradient descent (SGD)

```
ALGORITHM 2: Stochastic Gradient Descent (per-example)

Input:   per-example gradients ∇ℓ_i, learning rate η, initial θ₀, epochs E.
Output:  approximate minimiser.

1.  for epoch e = 1, …, E:
2.      π ← random permutation of {1, …, n}
3.      for i in π:
4.          g ← ∇ℓ_i(θ)                ▷ unbiased: 𝔼_i[ g ] = n · ∇L(θ)
5.          θ ← θ − η · g
6.  return θ
```

Per step: $\Theta(p)$ work, versus $\Theta(np)$ for Algorithm 1. The gradient estimate $g$ is **unbiased** but **noisy** — each step is wrong in direction even at the optimum. Consequence: with a constant $\eta$, SGD does **not** converge to a single point; the iterates *jitter* around $\theta^*$ inside a ball whose radius scales with $\eta$. Annealing the learning rate (e.g. $\eta_k = c / \sqrt{k}$) recovers convergence to $\theta^*$; this is the Robbins–Monro condition.

### 6.2 Mini-batch SGD

A compromise: average the gradient over a small batch of size $B$ (typically 32, 64, 128, …).

```
ALGORITHM 3: Mini-batch SGD

Input:   batch size B, learning rate η, initial θ₀, epochs E.
Output:  approximate minimiser.

1.  for epoch e = 1, …, E:
2.      shuffle {1, …, n} and split into ⌈n/B⌉ batches  B_1, …, B_K
3.      for each batch  B_k:
4.          g ← (1 / |B_k|) · Σ_{i ∈ B_k} ∇ℓ_i(θ)
5.          θ ← θ − η · g
6.  return θ
```

Mini-batch SGD keeps most of SGD's per-step cost savings while reducing variance by a factor of $B$. It is also vector- and GPU-friendly: $\nabla \ell$ for a whole batch is a single matrix operation.

In [ ]:
# Compare batch GD (B = n), mini-batch SGD (B = 10), and pure SGD (B = 1) on the same OLS problem.
def run(batch_size, n_passes, eta, seed=0):
    rng_local = np.random.default_rng(seed)
    theta = np.zeros(2)
    curve = [mse(theta)]
    for _ in range(n_passes):
        order = rng_local.permutation(n)
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            X_b, y_b = X[idx], y[idx]
            g = (2.0 / len(idx)) * X_b.T @ (X_b @ theta - y_b)
            theta = theta - eta * g
            curve.append(mse(theta))
    return theta, np.array(curve)

theta_batch, loss_batch = run(batch_size=n,  n_passes=10, eta=0.05)
theta_mini,  loss_mini  = run(batch_size=10, n_passes=10, eta=0.05)
theta_sgd,   loss_sgd   = run(batch_size=1,  n_passes=10, eta=0.05)

print(f"Batch GD   B = {n:>3}  →  θ = {theta_batch},  total updates = {len(loss_batch)-1}")
print(f"Mini SGD   B = {10:>3}  →  θ = {theta_mini},  total updates = {len(loss_mini)-1}")
print(f"SGD        B = {1:>3}  →  θ = {theta_sgd},  total updates = {len(loss_sgd)-1}")

plt.figure(figsize=(7.5, 4))
plt.plot(loss_batch, label=f"batch GD (B={n})")
plt.plot(loss_mini,  label="mini-batch SGD (B=10)", alpha=0.85)
plt.plot(loss_sgd,   label="pure SGD (B=1)",        alpha=0.55)
plt.yscale("log")
plt.xlabel("update step"); plt.ylabel("MSE (log scale)")
plt.title("Batch size trades per-step cost against gradient noise")
plt.legend()
plt.show()

## 7. Practical levers

### 7.1 Feature scaling

Theorem 4.2 says GD's linear-rate constant is $(\kappa - 1) / (\kappa + 1)$, with $\kappa = \operatorname{cond}(X)^2$. If one feature has range $0$–$10^6$ and another $0$–$1$, $\operatorname{cond}(X)$ is huge and convergence crawls. **Standardise** the features (subtract column mean, divide by column standard deviation) *before* training — this typically pushes $\kappa$ down by several orders of magnitude. The intercept column is left untouched.

### 7.2 Stopping criteria

Common choices:

- **Gradient norm.** Stop when $\|\nabla L(\theta_k)\| \le \text{tol}$. Direct check of the first-order optimality condition.
- **Loss plateau.** Stop when the *relative* drop $|L(\theta_k) - L(\theta_{k-1})| / \max(1, |L(\theta_{k-1})|)$ is below tol for several consecutive steps.
- **Validation early stopping.** When training loss is not the quantity of interest (e.g. supervised learning with overfitting risk), monitor a held-out loss and stop when it plateaus or worsens. This belongs to `04_statistics.ipynb`.

### 7.3 Beyond plain GD

Modern optimisers add memory (**momentum**, Nesterov acceleration) and per-coordinate scaling (Adagrad, RMSProp, **Adam**) to plain GD. They are out of scope here, but each shares the same skeleton (2.1): subtract a direction derived from past gradients, scaled by a step size. The convergence theory of §4 extends to them with appropriate adjustments to $L_{\text{smooth}}$ and the step-size analysis.

## 7. Practical levers

### 7.1 Feature scaling

Theorem 4.2 says GD's linear-rate constant is $(\kappa - 1) / (\kappa + 1)$, with $\kappa = \text{cond}(X)^2$. If one feature has range $0$–$10^6$ and another $0$–$1$, $\text{cond}(X)$ is huge and convergence crawls. **Standardise** the features (subtract column mean, divide by column standard deviation) *before* training — this typically pushes $\kappa$ down by several orders of magnitude. The intercept column is left untouched.

### 7.2 Stopping criteria

Common choices:

- **Gradient norm.** Stop when $\|\nabla L(\theta_k)\| \le \text{tol}$. Direct check of the first-order optimality condition.
- **Loss plateau.** Stop when the *relative* drop $|L(\theta_k) - L(\theta_{k-1})| / \max(1, |L(\theta_{k-1})|)$ is below tol for several consecutive steps.
- **Validation early stopping.** When training loss is not the quantity of interest (e.g. supervised learning with overfitting risk), monitor a held-out loss and stop when it plateaus or worsens. This belongs to `04_statistics.ipynb`.

### 7.3 Beyond plain GD

Modern optimisers add memory (**momentum**, Nesterov acceleration) and per-coordinate scaling (Adagrad, RMSProp, **Adam**) to plain GD. They are out of scope here, but each shares the same skeleton (2.1): subtract a direction derived from past gradients, scaled by a step size. The convergence theory of §4 extends to them with appropriate adjustments to $L_{\text{smooth}}$ and the step-size analysis.

## Takeaway

- **Why iterate.**   Closed form (4.2) is $\Theta(p^3)$ and MSE-only. GD is $\Theta(np)$ per step and works for any differentiable loss.
- **Update rule.**   $\theta_{k+1} = \theta_k - \eta \cdot \nabla L(\theta_k)$. For OLS specifically, $\nabla L(\theta) = \frac{2}{n} X^\top (X\theta - y)$.
- **Convergence (Theorem 4.2).**   Convex + $L$-smooth → $O(1/k)$ at $\eta = 1 / L_{\text{smooth}}$. Strongly convex with condition number $\kappa = \operatorname{cond}(X)^2$ → linear rate $\left(\frac{\kappa - 1}{\kappa + 1}\right)^k$.
- **Learning rate.**   Safe range $\eta \in (0, 2 / L_{\text{smooth}}]$. Too small → slow. Too large → diverge. Tune on a log scale.
- **Stochastic variants.**   SGD: $\Theta(p)$ per step, noisy, needs an $\eta$-schedule for convergence. Mini-batch SGD: best of both worlds, GPU-friendly.
- **Practical levers.**   Feature scaling reduces $\kappa$ → faster convergence. Stop on gradient norm or loss plateau.

Next: `04_statistics.ipynb` — having minimised the loss, what can we say about $\theta^*$ as a *random* quantity? Bias, variance, confidence intervals, hypothesis tests.